In [ ]:
# Cell 1 — Install dependencies
%pip install ollama tqdm -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2 — Imports
import email
import getpass
import imaplib
import os
import re
from datetime import datetime
from email.header import decode_header
from email.utils import parsedate_to_datetime

import ollama
from tqdm.notebook import tqdm

print("Ollama models currently available locally:")
for m in ollama.list().get("models", []):
    print(" -", m.get("model", m))


Ollama models currently available locally:
 - phi3:latest


In [ ]:
# Cell 3 — Configuration

IMAP_SERVER = "imap.gmail.com"
IMAP_PORT = 993

NUM_EMAILS = 5              
MAILBOX = "INBOX"           
OLLAMA_MODEL = "phi3"       
MAX_BODY_CHARS = 2000       
OUTPUT_DIR = "email_digests"

os.makedirs(OUTPUT_DIR, exist_ok=True)


EMAIL_ADDRESS = input("Gmail address: ").strip()

APP_PASSWORD = ""
while not APP_PASSWORD:
    APP_PASSWORD = getpass.getpass(
        "App Password (16-char, from myaccount.google.com/apppasswords): "
    ).strip().replace(" ", "")
    if not APP_PASSWORD:
        print("⚠️  No password captured — this can happen if the notebook's input prompt "
              "didn't render properly. Trying again with a visible prompt instead:")
        APP_PASSWORD = input("App Password (will be visible — clear this cell's output after): ").strip().replace(" ", "")

assert EMAIL_ADDRESS, "EMAIL_ADDRESS is empty — re-run this cell and make sure to type your address."
assert APP_PASSWORD, "APP_PASSWORD is empty — re-run this cell and make sure to type the App Password."
print(f"Captured email ({len(EMAIL_ADDRESS)} chars) and password ({len(APP_PASSWORD)} chars). Ready.")


Captured email (29 chars) and password (16 chars). Ready.


In [ ]:
# Cell 4 — Connecting to Gmail and fetch raw messages

def connect_readonly(email_address: str, app_password: str, mailbox: str = MAILBOX):
    imap = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
    imap.login(email_address, app_password)
    imap.select(mailbox, readonly=True)  
    return imap


def fetch_latest_email_ids(imap, count: int):
    status, data = imap.search(None, "ALL")
    if status != "OK":
        raise RuntimeError("IMAP search failed")
    all_ids = data[0].split()
    return all_ids[-count:] if len(all_ids) > count else all_ids


def decode_mime_header(raw_header) -> str:
    if not raw_header:
        return ""
    parts = decode_header(raw_header)
    decoded = ""
    for text, enc in parts:
        if isinstance(text, bytes):
            decoded += text.decode(enc or "utf-8", errors="replace")
        else:
            decoded += text
    return decoded


def get_plain_text_body(msg) -> str:
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            disposition = str(part.get("Content-Disposition", ""))
            if content_type == "text/plain" and "attachment" not in disposition:
                try:
                    charset = part.get_content_charset() or "utf-8"
                    return part.get_payload(decode=True).decode(charset, errors="replace")
                except Exception:
                    continue
       
        for part in msg.walk():
            if part.get_content_type() == "text/html":
                try:
                    charset = part.get_content_charset() or "utf-8"
                    html = part.get_payload(decode=True).decode(charset, errors="replace")
                    return re.sub("<[^<]+?>", " ", html)
                except Exception:
                    continue
        return ""
    else:
        try:
            charset = msg.get_content_charset() or "utf-8"
            return msg.get_payload(decode=True).decode(charset, errors="replace")
        except Exception:
            return msg.get_payload()


def fetch_emails(email_address: str, app_password: str, count: int = NUM_EMAILS) -> list:
    imap = connect_readonly(email_address, app_password)
    try:
        email_ids = fetch_latest_email_ids(imap, count)
        emails = []
        for eid in reversed(email_ids):  # newest first
            status, msg_data = imap.fetch(eid, "(RFC822)")
            if status != "OK":
                continue
            raw_email = msg_data[0][1]
            msg = email.message_from_bytes(raw_email)

            subject = decode_mime_header(msg.get("Subject"))
            sender = decode_mime_header(msg.get("From"))
            date_raw = msg.get("Date")
            try:
                date_parsed = parsedate_to_datetime(date_raw)
            except Exception:
                date_parsed = None

            body = get_plain_text_body(msg)
            body = re.sub(r"\s+", " ", body).strip()

            emails.append({
                "subject": subject or "(no subject)",
                "sender": sender or "(unknown sender)",
                "date": date_parsed.strftime("%Y-%m-%d %H:%M") if date_parsed else (date_raw or "unknown"),
                "body": body[:MAX_BODY_CHARS],
            })
        return emails
    finally:
        imap.logout()


In [ ]:
# Cell 5 — Running the fetch
raw_emails = fetch_emails(EMAIL_ADDRESS, APP_PASSWORD, NUM_EMAILS)

print(f"Fetched {len(raw_emails)} emails:\n")
for e in raw_emails:
    print(f"- [{e['date']}] {e['subject']}  (from {e['sender']})")


Fetched 5 emails:

- [2026-08-28 05:48] وظائف Go to Market Strategy Intern جديدة بنظام العمل عن بُعد  (from تنبيهات الوظائف على LinkedIn<jobalerts-noreply@linkedin.com>)
- [2026-08-28 03:03] AI is changing education. Universities must change too.  (from Huawei عبر LinkedIn<newsletters-noreply@linkedin.com>)
- [2026-08-27 14:57] Save 40% and learn from Microsoft experts   (from "Coursera" <Coursera@m.learn.coursera.org>)
- [2026-08-27 13:29] Last days: Free Access ends Sunday  (from The DataCamp Team <team@datacamp.com>)
- [2026-08-27 05:48] وظائف Recruiting Intern جديدة بدوام بنظام مختلط بين العمل عن بُعد والعمل من مقر الشركة  (from تنبيهات الوظائف على LinkedIn<jobalerts-noreply@linkedin.com>)


In [6]:
# Cell 6 — LLM summarization

SUMMARY_SYSTEM_PROMPT = """You are an assistant that summarizes emails concisely for a busy
person. For the email given, respond with:

1. A one-sentence summary of what it's about.
2. Up to 3 bullet points of key details (only if there's real substance — skip trivial emails).
3. A single line: "Action needed: <what, or 'None'>"

Keep it brief and skip filler. Do not invent details that aren't in the email."""


def summarize_email(e: dict, model: str = OLLAMA_MODEL) -> str:
    user_content = f"""From: {e['sender']}
Date: {e['date']}
Subject: {e['subject']}

Body:
{e['body']}"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        options={
            "temperature": 0.2,
            "num_predict": 200,   # caps response length so it can't ramble — big speed win
        },
    )
    return response["message"]["content"].strip()


In [ ]:
# Cell 7 —  summarization over all fetched emails
summaries = []
for e in tqdm(raw_emails, desc="Summarizing emails"):
    summary_text = summarize_email(e)
    summaries.append({**e, "summary": summary_text})

for s in summaries:
    print(f"=== {s['subject']} ===")
    print(f"From: {s['sender']}  |  {s['date']}")
    print(s['summary'])
    print()


Summarizing emails:   0%|          | 0/5 [00:00<?, ?it/s]

=== وظائف Go to Market Strategy Intern جديدة بنظام العمل عن بُعد ===
From: تنبيهات الوظائف على LinkedIn<jobalerts-noreply@linkedin.com>  |  2026-08-28 05:48
- The email announces new Go to Market Strategy Intern positions in Cairo, Egypt, with a focus on a Graphics & Visual Communication Intern role.
  - Internship in Cairo, Egypt for Go to Market Strategy, specifically Graphics & Visual Communication.
  - Application via personal profile and resume upload on LinkedIn.
  - Action needed: Submit application through LinkedIn.


-

=== AI is changing education. Universities must change too. ===
From: Huawei عبر LinkedIn<newsletters-noreply@linkedin.com>  |  2026-08-28 03:03
- Huawei discusses the transformative impact of AI on education, emphasizing the need for universities to adapt their teaching models and focus on skills relevant in a data-driven world.
- The article highlights the importance of mentorship, critical thinking, and problem-solving skills, which AI cannot easily replicat

In [8]:
# Cell 8 — Save the digest as a Markdown file

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
digest_path = os.path.join(OUTPUT_DIR, f"email_digest_{timestamp}.md")

with open(digest_path, "w", encoding="utf-8") as f:
    f.write(f"# Email Digest\n")
    f.write(f"_Generated {datetime.now().strftime('%Y-%m-%d %H:%M')} using local model `{OLLAMA_MODEL}`_\n\n")
    for s in summaries:
        f.write(f"## {s['subject']}\n")
        f.write(f"**From:** {s['sender']}  \n**Date:** {s['date']}\n\n")
        f.write(f"{s['summary']}\n\n---\n\n")

print(f"Saved digest to: {digest_path}")


Saved digest to: email_digests\email_digest_20260828_141536.md
